# Retail Sales Data Engineering Project using Python

This project uses Python and pandas to perform real-world retail transaction analysis.

## Project Goal

Build a mini data engineering and analytics pipeline:

1. Download dataset from Kaggle
2. Load raw CSV data
3. Inspect schema and data quality
4. Clean and transform data
5. Perform business analysis
6. Build KPI, customer summary, and RFM tables
7. Detect suspicious transactions
8. Export final analytical outputs

## Skills Covered

- Python basics for data engineering
- pandas DataFrame operations
- filtering
- groupby aggregations
- sorting
- date handling
- feature engineering
- customer-level aggregation
- RFM analysis
- anomaly detection
- CSV export

## 1. Install and Configure Kaggle API

Run this in Google Colab.

Before running this section:

1. Go to Kaggle
2. Open Account Settings
3. Create API Token
4. Upload `kaggle.json` when prompted

In [ ]:
!pip install -q kaggle

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 2. Download Dataset from Kaggle

Dataset used:

`rohitsahoo/sales-forecasting`

In [ ]:
!kaggle datasets download -d rohitsahoo/sales-forecasting
!unzip -o sales-forecasting.zip

## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

## 4. Load Data

Change the file name if your downloaded CSV has a different name.

In [ ]:
df = pd.read_csv("train.csv")

df.head()

## 5. Basic Data Inspection

This is the first step in every data engineering task.

In [ ]:
print("Rows and Columns:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types and Nulls:")
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## 6. Standardize Column Names

Clean column names make coding easier.

Example:

`Order ID` becomes `order_id`

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns.tolist()

## 7. Clean Data

Handle nulls, duplicates, and date conversion.

In [ ]:
df = df.drop_duplicates()

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")

df.isnull().sum()

## 8. Create Date Features

These are useful for reporting and partition-style analysis.

In [ ]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_month_name"] = df["order_date"].dt.month_name()
df["order_day"] = df["order_date"].dt.day
df["order_weekday"] = df["order_date"].dt.day_name()

df[["order_date", "order_year", "order_month", "order_month_name", "order_weekday"]].head()

# Business Analysis Tasks

## 1. Top Selling Categories

Business question:

Which product categories generate the highest revenue?

In [ ]:
top_categories = (
    df.groupby("category")["sales"]
    .sum()
    .reset_index()
    .sort_values(by="sales", ascending=False)
)

top_categories

## 2. Revenue by City

Business question:

Which cities generate the highest sales?

In [ ]:
city_revenue = (
    df.groupby("city")["sales"]
    .sum()
    .reset_index()
    .sort_values(by="sales", ascending=False)
)

city_revenue.head(10)

## 3. Ship Mode Analysis

Business question:

Which shipping mode is most used?

In [ ]:
ship_mode_usage = (
    df["ship_mode"]
    .value_counts()
    .reset_index()
)

ship_mode_usage.columns = ["ship_mode", "order_count"]

ship_mode_usage["percentage"] = (
    ship_mode_usage["order_count"] / ship_mode_usage["order_count"].sum()
) * 100

ship_mode_usage

## 4. Segment-wise Sales

Business question:

Which customer segment generates highest revenue?

In [ ]:
segment_sales = (
    df.groupby("segment")
    .agg(
        total_sales=("sales", "sum"),
        total_orders=("order_id", "nunique"),
        avg_order_value=("sales", "mean")
    )
    .reset_index()
    .sort_values(by="total_sales", ascending=False)
)

segment_sales

## 5. Discount Impact Analysis

This dataset may not contain discount in some Kaggle versions.

If `discount` is available, this section will run.

In [ ]:
if "discount" in df.columns:
    df["discount_bucket"] = pd.cut(
        df["discount"],
        bins=[-0.01, 0.10, 0.20, 0.30, 0.50, 1.00],
        labels=["0-10%", "10-20%", "20-30%", "30-50%", "50%+"]
    )

    discount_analysis = (
        df.groupby("discount_bucket")
        .agg(
            total_sales=("sales", "sum"),
            avg_sales=("sales", "mean"),
            total_orders=("order_id", "nunique")
        )
        .reset_index()
    )

    display(discount_analysis)
else:
    print("Discount column not available in this dataset.")

## 6. Monthly Sales Trend

Business question:

How does revenue move month by month?

In [ ]:
monthly_sales = (
    df.groupby(["order_year", "order_month"])
    .agg(total_sales=("sales", "sum"))
    .reset_index()
    .sort_values(by=["order_year", "order_month"])
)

monthly_sales.head()

## 7. State-wise Sales

Business question:

Which states generate highest revenue?

In [ ]:
state_sales = (
    df.groupby("state")["sales"]
    .sum()
    .reset_index()
    .sort_values(by="sales", ascending=False)
)

state_sales.head(10)

## 8. Sub-Category Performance

Business question:

Which sub-categories generate highest sales?

In [ ]:
subcategory_sales = (
    df.groupby("sub_category")["sales"]
    .sum()
    .reset_index()
    .sort_values(by="sales", ascending=False)
)

subcategory_sales

## 9. KPI Table

This table gives business-level summary metrics.

In [ ]:
kpi_table = pd.DataFrame({
    "metric": [
        "total_revenue",
        "total_orders",
        "unique_customers",
        "avg_order_value",
        "total_quantity"
    ],
    "value": [
        df["sales"].sum(),
        df["order_id"].nunique(),
        df["customer_id"].nunique(),
        df["sales"].mean(),
        df["quantity"].sum()
    ]
})

kpi_table

## 10. Customer Summary Table

This creates one row per customer.

This is similar to creating a customer mart in a data warehouse.

In [ ]:
customer_summary = (
    df.groupby("customer_id")
    .agg(
        customer_name=("customer_name", "first"),
        total_orders=("order_id", "nunique"),
        total_spend=("sales", "sum"),
        avg_order_value=("sales", "mean"),
        total_quantity=("quantity", "sum"),
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max")
    )
    .reset_index()
    .sort_values(by="total_spend", ascending=False)
)

customer_summary.head(10)

## 11. RFM Analysis

RFM means:

- Recency: how recently the customer purchased
- Frequency: how many orders the customer placed
- Monetary: how much the customer spent

In [ ]:
reference_date = df["order_date"].max()

rfm = (
    df.groupby("customer_id")
    .agg(
        customer_name=("customer_name", "first"),
        recency=("order_date", lambda x: (reference_date - x.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("sales", "sum")
    )
    .reset_index()
)

rfm.head()

In [ ]:
rfm["recency_score"] = pd.qcut(
    rfm["recency"],
    q=4,
    labels=[4, 3, 2, 1],
    duplicates="drop"
)

rfm["frequency_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
    duplicates="drop"
)

rfm["monetary_score"] = pd.qcut(
    rfm["monetary"],
    q=4,
    labels=[1, 2, 3, 4],
    duplicates="drop"
)

rfm["rfm_score"] = (
    rfm["recency_score"].astype(str)
    + rfm["frequency_score"].astype(str)
    + rfm["monetary_score"].astype(str)
)

rfm.head()

## 12. Suspicious Transaction Detection

Examples:

- very high sales amount
- very high quantity
- sales in top 1 percentile

In [ ]:
sales_99_percentile = df["sales"].quantile(0.99)
quantity_99_percentile = df["quantity"].quantile(0.99)

suspicious_transactions = df[
    (df["sales"] > sales_99_percentile) |
    (df["quantity"] > quantity_99_percentile)
]

suspicious_transactions.head()

# Export Final Outputs

These files can be uploaded to GitHub as project outputs.

In [ ]:
top_categories.to_csv("top_categories.csv", index=False)
city_revenue.to_csv("city_revenue.csv", index=False)
segment_sales.to_csv("segment_sales.csv", index=False)
monthly_sales.to_csv("monthly_sales.csv", index=False)
state_sales.to_csv("state_sales.csv", index=False)
subcategory_sales.to_csv("subcategory_sales.csv", index=False)
kpi_table.to_csv("kpi_table.csv", index=False)
customer_summary.to_csv("customer_summary.csv", index=False)
rfm.to_csv("rfm_analysis.csv", index=False)
suspicious_transactions.to_csv("suspicious_transactions.csv", index=False)

print("All output files created successfully.")

# Project Summary

In this project, we built a complete mini data engineering and analytics workflow.

## What was done

1. Downloaded data from Kaggle
2. Loaded raw CSV data into pandas
3. Inspected schema, nulls, and duplicates
4. Standardized column names
5. Created date-based features
6. Built business analysis tables
7. Created KPI table
8. Created customer summary table
9. Built RFM analysis
10. Detected suspicious transactions
11. Exported final CSV outputs

## Concepts Practiced

- Data ingestion
- Data cleaning
- Data transformation
- Aggregation
- Feature engineering
- Customer analytics
- RFM modeling
- Anomaly detection
- Exporting curated data

# GitHub README Text

Copy this section into your GitHub README.

## Retail Sales Data Engineering Project

This project demonstrates a complete beginner-friendly data engineering and analytics workflow using Python, pandas, Google Colab, and a Kaggle retail sales dataset.

### Objectives

- Ingest retail transaction data from Kaggle
- Clean and standardize raw data
- Perform exploratory data analysis
- Build business KPI tables
- Create customer-level summary tables
- Perform RFM customer segmentation
- Detect suspicious transactions
- Export final analytics-ready CSV files

### Tools Used

- Python
- pandas
- Google Colab
- Kaggle API

### Key Outputs

- Top category sales
- City-wise revenue
- Segment-wise performance
- Monthly sales trend
- KPI table
- Customer summary table
- RFM analysis
- Suspicious transaction report

### Skills Demonstrated

- Data ingestion
- Data cleaning
- Data transformation
- Grouped aggregations
- Date feature engineering
- Customer analytics
- CSV export pipeline